# Cardiac Patient Monitoring System

## 04 — Model Evaluation

## Objective

Evaluate the candidate classifiers using cross-validation and final test-set metrics. Interpret the confusion matrix, false positives, and false negatives.

## Imports and Load Data

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

DATA_PATH = DATA_DIR / "cardio_train.csv"

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve
)

df = pd.read_csv(DATA_DIR / "cardio_clean.csv")
X = df.drop(columns=["cardio"])
y = df["cardio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    )
}


## Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"Model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).set_index("Model")
display(cv_results.round(4))
cv_results.to_csv(OUTPUT_DIR / "cross_validation_results.csv")


## Final Test Evaluation

In [ ]:
test_rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

test_results = pd.DataFrame(test_rows).set_index("Model")
display(test_results.round(4))
test_results.to_csv(OUTPUT_DIR / "final_test_results.csv")


## Confusion Matrix — Best Model by ROC-AUC

In [ ]:
best_name = test_results["ROC-AUC"].idxmax()
best_model = models[best_name]
best_model.fit(X_train, y_train)

best_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_pred)

print("Selected model:", best_name)
print("\nClassification report:")
print(classification_report(y_test, best_pred, digits=4))

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_best_model.png", dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")


## False Positives and False Negatives

- **False Positive:** the model predicts the positive class when the actual target is negative.
- **False Negative:** the model predicts the negative class when the actual target is positive.

In this educational project, these terms describe classification errors only; they are not clinical decisions.

## ROC Curves

In [ ]:
plt.figure(figsize=(7,5))

for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png", dpi=150)
plt.show()


## Evaluation Conclusion

Model selection should consider the complete metric profile and cross-validation stability rather than a single score. The selected model will be reused in the pipeline notebook.